# IDAP 2026 camera-ready — Colab #1: sentinel cleanup + full re-evaluation

Runs the *mandatory* part of the revision:

1. **Gate check** — is the archived `article` field the full source, or a truncated model input?
2. **Clean** every archived prediction with the exact regex `src/student/infer.py` uses.
3. **Prove** post-hoc stripping == regenerating with the fixed decoder.
4. **Re-score** all systems on both the dirty and clean predictions (ROUGE P/R/F + BERTScore + error flags), with per-example scores so paired CIs are possible.
5. **Artifact impact table** — how much the sentinel contamination moved each published number.

Expected wall time: **~90–120 min** on a T4, most of it BERTScore. Safe to re-run; steps are idempotent.

> Runtime → Change runtime type → **T4 GPU** before you start.


In [ ]:
# --- 0. environment ---
import os, subprocess, sys, json, shutil, pathlib, time
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv,noheader'],
                     capture_output=True, text=True).stdout.strip() or 'NO GPU — stop and switch runtime to T4')
from google.colab import drive
drive.mount('/content/drive')

DRIVE = '/content/drive/MyDrive/ceng467_termproject/CENG467-term-project'
assert os.path.isdir(DRIVE), f'Drive repo not found at {DRIVE}'
print('Drive predictions:', len(os.listdir(f'{DRIVE}/outputs/predictions')), 'entries')


In [ ]:
# --- 1. fresh clone of the revision branch (fast local disk, not Drive) ---
REPO = '/content/repo'
BRANCH = 'revision-v2'
URL = 'https://github.com/cagancaliskan/turkish-summarization-distillation.git'
if os.path.isdir(REPO):
    !cd {REPO} && git fetch origin {BRANCH} -q && git checkout {BRANCH} -q && git pull -q
else:
    !git clone -q --branch {BRANCH} {URL} {REPO}
os.chdir(REPO)
!git log --oneline -1

!pip install -q 'rouge-score>=0.1.2' 'bert-score>=0.3.13' 'peft>=0.13.0' 2>&1 | tail -2
import numpy, scipy; print('numpy', numpy.__version__, '| scipy', scipy.__version__)
!python src/eval/stats.py

# Colab ships torchao 0.10; peft's LoRA dispatcher calls is_torchao_available()
# and RAISES when the version is below its minimum instead of skipping the
# backend. This project uses no quantized adapters, so removing torchao is the
# clean fix -- upgrading it risks dragging a different torch build along.
!pip uninstall -y -q torchao 2>&1 | tail -1
import subprocess as _sp
_chk = _sp.run(['python','-c',
    'from peft.import_utils import is_torchao_available as f; print("torchao gate:", f())'],
    capture_output=True, text=True)
print(_chk.stdout.strip() or _chk.stderr.strip()[-300:])
assert 'torchao gate: False' in _chk.stdout, 'torchao still blocking peft — tell me before continuing'


In [ ]:
# --- 2. bring the archived v1 predictions onto local disk (read-only snapshot) ---
V1 = f'{REPO}/outputs/predictions/v1_raw'
os.makedirs(V1, exist_ok=True)
src_dir = f'{DRIVE}/outputs/predictions'
n = 0
for f in sorted(os.listdir(src_dir)):
    if f.endswith('.jsonl'):
        dst = f'{V1}/{f}'
        if not os.path.exists(dst):
            shutil.copy2(f'{src_dir}/{f}', dst)
        n += 1
print(f'{n} prediction files staged at {V1}')
!ls -la {V1} | head -25


## STEP 1 — Gate check: is `article` the full source text?

Every faithfulness metric is computed against this field. If it is the truncated model input, grounding checks silently re-create the 600-character judging bug at scale — and they bias *against* the verbose teachers, i.e. against exactly the systems the hallucination claim compares to.

**If this prints `ACTION:`, stop and tell me before running anything downstream.**


In [ ]:
!python scripts/audit_articles.py --in-dir outputs/predictions/v1_raw \
    --out-json outputs/results/v2/article_audit.json


## STEP 2 — Strip the sentinels

Per-system incidence is itself a result: it is the rate at which each model drops back into mT5's span-corruption mode. It goes in the paper.


In [ ]:
!python -m src.eval.clean_predictions \
    --in-dir  outputs/predictions/v1_raw \
    --out-dir outputs/predictions/clean \
    --stats-json outputs/results/v2/sentinel_incidence.json


## STEP 3 — Prove that post-processing == regenerating

Generates 200 articles twice from the same checkpoint — once with `--keep-sentinels`, once without — and checks `regex_strip(dirty) == clean` for every row. This is what lets us re-score archived predictions instead of re-running inference for all 18 systems.

~6 minutes.


In [ ]:
VER = f'{REPO}/outputs/predictions/_verify'
os.makedirs(VER, exist_ok=True)

# 200-row input built from the archived file itself (id + article + reference)
src = f'{REPO}/outputs/predictions/v1_raw/S_gpt.jsonl'
with open(src, encoding='utf-8') as fi, open(f'{VER}/input200.jsonl','w',encoding='utf-8') as fo:
    for i, line in enumerate(fi):
        if i >= 200: break
        r = json.loads(line)
        fo.write(json.dumps({'id': r['id'], 'article': r['article'],
                             'reference': r.get('reference')}, ensure_ascii=False) + '\n')
print('built input200.jsonl')

CKPT_SRC = f'{DRIVE}/outputs/checkpoints/S_gpt_n10000_r8/final'
CKPT = '/content/ckpt_S_gpt_r8'
if not os.path.isdir(CKPT):
    shutil.copytree(CKPT_SRC, CKPT)
print('checkpoint files:', os.listdir(CKPT))


In [ ]:
!python -m src.student.infer --model-path {CKPT} \
    --input outputs/predictions/_verify/input200.jsonl \
    --out   outputs/predictions/_verify/dirty.jsonl --keep-sentinels

!python -m src.student.infer --model-path {CKPT} \
    --input outputs/predictions/_verify/input200.jsonl \
    --out   outputs/predictions/_verify/clean.jsonl

!python scripts/verify_sentinel_equivalence.py \
    --dirty    outputs/predictions/_verify/dirty.jsonl \
    --clean    outputs/predictions/_verify/clean.jsonl \
    --archived outputs/predictions/v1_raw/S_gpt.jsonl \
    --out-json outputs/results/v2/sentinel_equivalence.json


## STEP 4 — Re-score everything

Both arms (dirty = what the paper reports, clean = corrected) so the impact is measurable. Per-example scores are written out — paired bootstrap CIs and paired significance tests need them.

BERTScore is batch-invariant, so `--bertscore-batch 16` gives identical numbers to the published batch of 8, just faster.


In [ ]:
MAIN = {'B1':'B1_zeroshot','B2':'B2_human','B3a':'B3a_gpt','B3b':'B3b_claude',
        'S-gpt':'S_gpt','S-claude':'S_claude'}
OOD  = {'B1':'ood_B1','B2':'ood_B2','B3a':'ood_B3a','B3b':'ood_B3b',
        'S-gpt':'ood_S_gpt','S-claude':'ood_S_claude'}
ABL  = {'S-gpt-n1k':'S_gpt_n1k','S-gpt-n5k':'S_gpt_n5k','S-gpt-r4':'S_gpt_r4',
        'S-gpt-r16':'S_gpt_r16','S-gpt-r32':'S_gpt_r32','S-gpt-detailed1k':'S_gpt_detailed_1k'}

def preds(mapping, arm):
    d = 'v1_raw' if arm == 'dirty' else 'clean'
    return ' '.join(f'--pred {k}=outputs/predictions/{d}/{v}.jsonl' for k, v in mapping.items())

def run_eval(mapping, arm, tag, bertscore=True):
    metrics = 'rouge errors bertscore' if bertscore else 'rouge errors'
    cmd = (f'python -m src.eval.run_eval_v2 {preds(mapping, arm)} '
           f'--metrics {metrics} --bertscore-batch 16 '
           f'--per-example-dir outputs/results/v2/per_example_{arm}_{tag} '
           f'--out-json outputs/results/v2/{tag}_eval_{arm.upper()}.json')
    print('>>>', tag, arm); t0 = time.time()
    rc = subprocess.run(cmd, shell=True).returncode
    print(f'   rc={rc}  {time.time()-t0:.0f}s')
    assert rc == 0, f'{tag}/{arm} failed'


In [ ]:
run_eval(MAIN, 'dirty', 'main', bertscore=True)
run_eval(MAIN, 'clean', 'main', bertscore=True)


In [ ]:
run_eval(OOD, 'dirty', 'ood', bertscore=True)
run_eval(OOD, 'clean', 'ood', bertscore=True)


In [ ]:
run_eval(ABL, 'dirty', 'abl', bertscore=False)
run_eval(ABL, 'clean', 'abl', bertscore=False)


## STEP 5 — Artifact impact table

`delta = clean - dirty`. A positive delta means the number **currently printed in the paper understates the system**.


In [ ]:
for tag in ['main','ood','abl']:
    print('='*78); print(tag.upper()); print('='*78)
    !python scripts/artifact_impact.py \
        --dirty outputs/results/v2/{tag}_eval_DIRTY.json \
        --clean outputs/results/v2/{tag}_eval_CLEAN.json \
        --per-example-dirty outputs/results/v2/per_example_dirty_{tag} \
        --per-example-clean outputs/results/v2/per_example_clean_{tag} \
        --out-json outputs/results/v2/artifact_impact_{tag}.json \
        --out-md   outputs/results/v2/artifact_impact_{tag}.md


## STEP 6 — Push results back to Drive

Everything under `outputs/results/v2/` and `outputs/predictions/clean/` is what the next steps read.


In [ ]:
dst = f'{DRIVE}/outputs'
for sub in ['results/v2', 'predictions/clean', 'predictions/_verify']:
    s, d = f'{REPO}/outputs/{sub}', f'{dst}/{sub}'
    os.makedirs(os.path.dirname(d), exist_ok=True)
    if os.path.isdir(d): shutil.rmtree(d)
    shutil.copytree(s, d)
    print('synced', sub)

print()
print('--- sentinel incidence ---')
inc = json.load(open(f'{REPO}/outputs/results/v2/sentinel_incidence.json', encoding='utf-8'))
for k, v in sorted(inc.items()):
    print(f"  {k:<26} {100*v['frac_with_sentinel']:5.1f}% of rows, "
          f"{v['n_sentinels_total']:5d} sentinels, {v['n_empty_after_clean']} emptied")
print()
print('DONE. Send me outputs/results/v2/*.md and *.json (or just tell me the numbers).')
